# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My lane: Refresh / Content Opportunity Scoring.**

1. **One row = one content page, for one client, aggregated over one calendar month** (March 2026 — a mid-panel month, not the sealed final month). I don't use the raw daily grain directly; I roll each page's daily rows up into one monthly summary row so I can compare "first half of the month" vs "second half of the month" for that page.
2. **Table(s):** `fact_content_daily_performance` (partition `month=2026-03`) joined to `dim_content` for static page attributes (word count, last-updated date).
3. **Time window:** March 1–31, 2026. Features come only from March 1–15; the label comes only from March 16–31 — no overlap.
4. **Label / proxy:** `is_declining` = did the page's impressions in the second half of March fall below its impressions in the first half? A real future-outcome label (built from the full ~17-month panel, prior 90 days → next 30 days) is the capstone upgrade; this month-split version is the honest v1 I can build and test right now, on real warehouse data, without waiting.
5. **Deliberately excluded:** `imp_second_half` itself (and anything computed from it) as a feature — it's what the label is made from, so including it would leak the answer. I prove this below instead of just asserting it.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `imp_first_half`, `clicks_first_half`, `avg_position_first_half` | **Feature** | measured Mar 1–15, known before the Mar 16–31 window I'm predicting |
| `word_count` | **Feature** | static page attribute, set well before March |
| `days_since_last_update` | **Feature** | derived from `content_updated_date` (dim_content) minus the decision date; known before |
| `imp_second_half` | **Label source** | the outcome window — `is_declining` is computed from it |
| `client_hash_id`, `content_hash_id` | **Context** | grouping/joining only, never a model input |
| `gsc_data_available` | **Context / filter** | used to keep only real rows, not fed to the model |
| `content_type`, `content_created_date` | **Context** | kept for reading/inspection, not used as features in this v1 |
| `imp_second_half` used as a feature | **Excluded (leakage)** | it's literally what the label is built from — proven in section 3 |
| any product decision flag (`health_score`, `priority_score`, ...) | **Excluded** | not shipped in this dataset by design, and even if rebuilt, never usable as a feature or label |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# Setup — connect DuckDB to the gated warehouse release.
import os, sys, getpass
import duckdb
import pandas as pd
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your HF READ token (hf_...): ")

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
print("Connected. Using month=2026-03 (mid-panel; NOT the sealed final month).")

Connected. Using month=2026-03 (mid-panel; NOT the sealed final month).


### Query 1 — grain: is one row really (report_date, client, content)?

**Named limitation: this is a within-month split, not a real future-outcome label.** Comparing March 1–15 to March 16–31 is a fast, honest way to test the leakage discipline and pipeline mechanics on real data — but a 15-day window can't tell real decline apart from a mid-month seasonality wiggle, a Google algorithm update, or a sibling page absorbing traffic (consolidation). It's also just one month, so a page having a bad or good two weeks isn't the same as a sustained trend.

**Other limits of this slice:** history depth differs wildly per client (`dim_clients.gsc_data_start` varies by over a year across the panel), so a page appearing "new" in March might just be a client whose tracking started late, not an actually new page. And `gsc_data_available IS TRUE` cut the raw month down from ~9.8M rows to ~3.6M (Query 3) — the remaining ~63% aren't zero-traffic, they're rows before that client's tracking existed, which I correctly excluded rather than counting as "no visibility."

**The upgrade for the capstone:** replace this two-week split with a genuine prior-90-days → next-30-days label built across the full panel, which needs the per-client `gsc_data_start` check this notebook flags but doesn't yet use.

### Query 2 — row count and date span for my slice (month=2026-03)

In [2]:
n, mn, mx = con.sql(f"""
    SELECT COUNT(*) n, MIN(report_date) mn, MAX(report_date) mx FROM {FACT}
""").fetchone()
print(f"Rows in month=2026-03: {n:,}")
print(f"Date span: {mn} to {mx}")
assert str(mn) == "2026-03-01" and str(mx) == "2026-03-31", "date span doesn't match the partition name"

Rows in month=2026-03: 9,841,378
Date span: 2026-03-01 to 2026-03-31


### Query 3 — availability: filter with `IS TRUE`, count what survives

Some rows exist in the fact table before a client's GSC tracking actually started — `gsc_data_available` distinguishes "genuinely zero" from "no tracking yet." I filter with `IS TRUE` (never `= TRUE` or `= 1`, in case of nulls) and show the drop.

In [3]:
total, available = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM {FACT}
""").fetchone()
print(f"Total rows:                 {total:,}")
print(f"Rows with gsc_data_available IS TRUE: {available:,} ({available/total:.1%})")
print(f"Rows dropped (no real GSC tracking yet): {total - available:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows:                 9,841,378
Rows with gsc_data_available IS TRUE: 3,611,061 (36.7%)
Rows dropped (no real GSC tracking yet): 6,230,317


### Five features, max — the feature frame

Roll March's daily rows up to one row per page: features from **Mar 1–15**, label from **Mar 16–31**. Minimum-volume filter (`imp_first_half >= 10`) so the decline ratio isn't pure noise on near-zero pages.

In [4]:
frame = con.sql(f"""
    WITH daily AS (
        SELECT * FROM {FACT} WHERE gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
            AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0
                     THEN gsc_avg_position END) AS avg_position_first_half
        FROM daily
        GROUP BY 1, 2
        HAVING imp_first_half >= 10
    )
    SELECT a.*, d.word_count, d.content_updated_date
    FROM agg a
    LEFT JOIN {DIM} d USING (client_hash_id, content_hash_id)
""").df()

CUTOFF = pd.Timestamp("2026-03-15")
frame["days_since_last_update"] = (CUTOFF - pd.to_datetime(frame["content_updated_date"])).dt.days
frame["ctr_first_half"] = (frame["clicks_first_half"] / frame["imp_first_half"]).clip(upper=1)
frame["word_count"] = frame["word_count"].fillna(0)
frame["is_declining"] = (frame["imp_second_half"] < frame["imp_first_half"]).astype(int)

print(f"Pages in feature frame: {len(frame):,}")
print(f"Declining rate (Mar 16-31 impressions < Mar 1-15 impressions): {frame['is_declining'].mean():.3f}")

FEATURES = ["imp_first_half", "clicks_first_half", "avg_position_first_half",
            "word_count", "days_since_last_update"]
frame[FEATURES + ["is_declining"]].head(5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages in feature frame: 120,513
Declining rate (Mar 16-31 impressions < Mar 1-15 impressions): 0.433


,imp_first_half,clicks_first_half,avg_position_first_half,word_count,days_since_last_update,is_declining
0,120.0,0.0,2.726381,0,-64,1
1,63.0,0.0,14.385119,0,-66,1
2,179.0,0.0,26.296183,0,-66,0
3,434.0,0.0,4.292451,0,-64,0
4,247.0,0.0,4.675600,0,-64,1


**The five features, and why each is knowable at the decision moment (March 15, before the label window starts):**

1. `imp_first_half` — impressions Mar 1–15. Available because it's fully in the past relative to the Mar 16 decision point.
2. `clicks_first_half` — clicks Mar 1–15. Same reason: measured before the window I'm predicting.
3. `avg_position_first_half` — average Google rank Mar 1–15 (rows with no position data excluded, not treated as rank 0). Measured before the cutoff.
4. `word_count` — from `dim_content`, a static attribute of the page set when it was written, long before March. Available at any decision moment.
5. `days_since_last_update` — Mar 15 minus `content_updated_date`. Computable the moment you stand at Mar 15 looking forward; doesn't need anything from the future.

### The trap — add one label-derived column on purpose

`is_declining` is computed by comparing `imp_second_half` to `imp_first_half`. If I hand the model `imp_second_half` itself as a feature, it isn't predicting the future anymore — it already has the future. Watch the score jump, then delete it.

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

X_safe = frame[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = frame["is_declining"].values

honest_scores = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=42),
                                 X_safe, y, cv=3, scoring="roc_auc")
print(f"HONEST — 5 safe features only — ROC-AUC: {honest_scores.mean():.3f}")

# --- deliberate leak: add the exact column the label is derived from ---
X_leaky = X_safe.copy()
X_leaky["imp_second_half"] = frame["imp_second_half"].values   # <-- the leak

leaky_scores = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=42),
                                X_leaky, y, cv=3, scoring="roc_auc")
print(f"LEAKY  — + imp_second_half (the label's own source) — ROC-AUC: {leaky_scores.mean():.3f}")
print(f"\nJump from leakage: {honest_scores.mean():.3f} -> {leaky_scores.mean():.3f} "
      f"(+{leaky_scores.mean() - honest_scores.mean():.3f})")

# --- delete the leak, keep the honest number ---
del X_leaky
print("\nLeaky feature removed. The number I report and build on is the honest one: "
      f"{honest_scores.mean():.3f} ROC-AUC on 5 pre-decision features.")

HONEST — 5 safe features only — ROC-AUC: 0.612


LEAKY  — + imp_second_half (the label's own source) — ROC-AUC: 0.740

Jump from leakage: 0.612 -> 0.740 (+0.129)

Leaky feature removed. The number I report and build on is the honest one: 0.612 ROC-AUC on 5 pre-decision features.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.